# 05 · Vector search with sentence-transformers

Keyword search (notebooks 01–04) matches **words**. Vector search matches **meaning**: each text becomes a 384-number vector with [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), and texts with similar meaning get similar vectors.

The chatbot uses the same model: `chatbot/scripts/build_embeddings.py` builds its document vectors.

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

## 1. Text → vector

In [2]:
vector = model.encode("Can I still join the course?", normalize_embeddings=True)
vector.shape, vector[:5]

((384,),
 array([ 0.00515177, -0.05652114, -0.00070674,  0.01608383,  0.04877055],
       dtype=float32))

## 2. Similar meaning → similar vectors

With normalised vectors, the dot product is the cosine similarity: 1 = same meaning, 0 = unrelated.

In [3]:
query = "When is the homework due?"
candidates = [
    "What is the submission deadline for assignments?",  # same meaning, no shared words
    "When is the homework party?",                       # shared words, different meaning
    "How do I install Docker on Windows?",               # unrelated
]

q = model.encode(query, normalize_embeddings=True)
for text, v in zip(candidates, model.encode(candidates, normalize_embeddings=True)):
    print(f"{q @ v:.3f}  {text}")

0.563  What is the submission deadline for assignments?
0.743  When is the homework party?
0.035  How do I install Docker on Windows?


The paraphrase scores far above the unrelated question, even though it shares no words with the query. But the *homework party* sentence still wins, because shared words pull vectors together in a small model. That's why the chatbot combines vector search **with** keyword search (hybrid).

## 3. Embed the FAQ

In [4]:
from llm_zoomcamp_dtc.ingest import load_faq_data

documents = load_faq_data(["llm-zoomcamp"])
texts = [f"{d['question']}\n{d['answer']}" for d in documents]
embeddings = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
embeddings.shape

(153, 384)

## 4. Vector search vs keyword search

In [5]:
from minsearch import Index, VectorSearch

vector_index = VectorSearch(keyword_fields=["course"])
vector_index.fit(embeddings, documents)

keyword_index = Index(text_fields=["question", "section", "answer"], keyword_fields=["course"])
keyword_index.fit(documents)


def compare(query, k=3):
    print(f"QUERY: {query}\n")
    print("keyword:")
    for d in keyword_index.search(query, boost_dict={"question": 2.0, "section": 0.5}, num_results=k):
        print("  -", d["question"])
    print("vector:")
    for d in vector_index.search(model.encode(query, normalize_embeddings=True), num_results=k):
        print("  -", d["question"])


compare("when is the capstone due?")

QUERY: when is the capstone due?

keyword:
  - When is the capstone project deadline?
  - Can I use Evidently for the capstone project?
  - Is there a minimum dataset size for the capstone project?
vector:
  - When is the capstone project deadline?
  - Can I use Evidently for the capstone project?
  - Does the capstone project need a completely original idea?


In [6]:
compare("is it too late to sign up?")

QUERY: is it too late to sign up?

keyword:
  - Is it a group project?
  - How to set up a new dlt project when loading from cloud?
  - uv says Failed to hardlink files in Codespaces. Is it an error?
vector:
  - Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
  - I just discovered the course. Can I still join?
  - How long do I have to complete the peer reviews once they're assigned?


## Takeaways

- Vectors find answers phrased differently from the question: *"is it too late to sign up?"* → *"Can I still join?"*, where keyword search returned unrelated entries.
- Keywords are still better for exact terms: error messages, library names, numbers.
- The chatbot runs both and merges the rankings with **reciprocal rank fusion**. On the evaluation set this raised hit rate@5 from 0.909 (keywords only) to 0.945.